- Pixel-Flips an unebenen Grenzbereichen (Wälder, Küsten, Farmland, etc.).
- Vertauschen von Labels, um Verwirrung durch Ähnlichkeit beim manuellen Labeln zu simulieren (z. B. farm/pasture, asphalt/building). 
- Rotation/Verschieben von zu labelnden Bildausschnitten oder Objekten,  um Missalignment im Labeling mit Tools wie OpenStreetMap zu 
- Shift/Verzerren von Objekten wie hohen Gebäuden, da je nach Einfallswinkel des Sensors unterschiedliche Seiten eingefangen werden.

In [1]:
import numpy as np
import scipy.ndimage as ndimage
import cv2
import sklearn as sk
from calculate_metrics import load_labels
from pathlib import Path
from inference import color_model_output, DATA_ROOT
from PIL import Image



#### Uneven Boundary Noise

1. Filter out pixels at boundaries
2. With (constantly declining) probability swap classes in (constant) radius around boundary pixel (**too complicated for beginning**)
3. ...

In [ ]:
def add_boundary_noise(mask, noise_probability=0.3, structure_size=3):
    """
    Erzeugt Rauschen ausschließlich an den Grenzen von Klassen.
    """
    # Finden der Grenzen: Wo unterscheidet sich die Ur-Maske von der dilatierten?
    dilated = ndimage.maximum_filter(mask, size=structure_size)
    eroded = ndimage.minimum_filter(mask, size=structure_size)
    boundary_mask = dilated != eroded
    
    # Zufällige Pixel innerhalb der Grenze auswählen
    random_noise = np.random.rand(*mask.shape) < noise_probability
    pixels_to_flip = boundary_mask & random_noise
    
    noisy_mask = mask.copy()
    # Weise den zu flippenden Pixeln den Wert der dilatierten Maske zu (Nachbarklasse)
    noisy_mask[pixels_to_flip] = dilated[pixels_to_flip]
    
    return noisy_mask

#### Probabilistic Class Swapping

1. Calculate Concfusion Matrix on original Predictions
2. Make this foundational swap matrix with probabilities for swapping classes
3. **Foreach Object** in every image **swap class** with **prob=swap_matrix**


In [ ]:
# with superpixels

from skimage.segmentation import slic


def add_semantic_swapping_superpixels(mask, relative_confusion_matrix,
                          n_segments=400, compactness=10, sigma=1.0):
    """
    mask: 2D y_pred label image
    relative_confusion_matrix: shape (n_classes, n_classes)
        row i gives probabilities of class i turning into class j
    image: optional image to compute superpixels on; if None, use mask itself
    """
    
    image = mask.astype(np.float32)

    segments = slic(image,
                    n_segments=n_segments,
                    compactness=compactness,
                    sigma=sigma,
                    start_label=1,
                    channel_axis=None)

    noisy_mask = mask.copy()
    n_classes = relative_confusion_matrix.shape[1]

    for segment_id in np.unique(segments):
        segment_mask = segments == segment_id
        if not segment_mask.any():
            continue

        segment_labels = mask[segment_mask]
        if segment_labels.size == 0:
            continue

        # choose the dominant class in this superpixel
        segment_class = np.bincount(segment_labels, minlength=n_classes).argmax()

        probs = relative_confusion_matrix[segment_class].astype(float)
        probs = probs / probs.sum()

        new_class = np.random.choice(np.arange(n_classes), p=probs)
        if new_class != segment_class:
            noisy_mask[segment_mask] = new_class

    return noisy_mask

In [ ]:
# with connected components

def add_semantic_swapping(y_true, y_pred, n_classes):
    """
    mask: 2D y_pred label image
    relative_confusion_matrix: shape (n_classes, n_classes),
        row i gives probabilities of class i turning into class j
    """

    structure_size=3
    confusion_mat = sk.metrics.confusion_matrix(y_true.flatten(), y_pred.flatten(), labels=range(n_classes))
    relative_confusion_matrix = confusion_mat / confusion_mat.sum(axis=1, keepdims=True)
    
    noisy_set = np.zeros_like(y_pred)

    for i, mask in enumerate(y_pred):

        noisy_mask = mask.copy()
        structure = np.ones((structure_size, structure_size), dtype=np.int32)
        n_classes = relative_confusion_matrix.shape[1]

        for class_id in np.unique(mask):
            if class_id < 0 or class_id >= n_classes:
                continue

            class_mask = (mask == class_id)
            if not class_mask.any():
                continue

            labeled, n_objects = ndimage.label(class_mask, structure=structure)
            probs = relative_confusion_matrix[class_id].astype(float)

            probs = probs / probs.sum()
            
            # with open(DATA_ROOT / "noise" / noise_method / "label_changes.txt", "a") as f:

            for obj_id in range(1, n_objects + 1):
                object_mask = (labeled == obj_id)
                new_class = np.random.choice(np.arange(n_classes), p=probs)
                # f.write("NEW CLASS: " + str(new_class)  + " for original class: " + str(class_id) + "\n")
                if new_class != class_id:
                    noisy_mask[object_mask] = new_class
        
        noisy_set[i] = noisy_mask

    return noisy_set

In [ ]:
def shift_class_original_fill(y_pred, class_id, shift_x=5, shift_y=5, shift_prob=1.0):
    """
    Verschiebt nur die Pixel einer bestimmten Klasse (z.B. Gebäude-Layover) mit Wahrscheinlichkeit shift_prob.
    Der unter dem verschobenen Gebäude liegende Bereich wird extrahiert, rotiert und in den leeren Bereich eingefügt.
    
    Parameters:
    - mask: 2D label image
    - class_id: Class to shift
    - shift_x, shift_y: Translation amounts
    - shift_prob: Probability of shifting each connected component (0-1)
    """
    
    noisy_set = np.zeros_like(y_pred)

    for i, mask in enumerate(y_pred):
    
        noisy_mask = mask.copy()
        structure = np.ones((3, 3), dtype=np.int32)
        
        # Find all connected components of the class
        class_mask = (mask == class_id)
        labeled, n_objects = ndimage.label(class_mask, structure=structure)
        

        for obj_id in range(1, n_objects + 1):
            # Shift only with probability shift_prob
            if np.random.rand() > shift_prob:
                continue
            
            object_mask = (labeled == obj_id).astype(np.float32)
            
            # Shift the object via affine transformation
            translation_matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
            shifted_object_mask = cv2.warpAffine(object_mask, translation_matrix, 
                                                (mask.shape[1], mask.shape[0]), 
                                                flags=cv2.INTER_NEAREST)
            
            original_region = object_mask == 1
            overshadowed_region = shifted_object_mask == 1
            
            # Extract the content that gets overshadowed
            overshadowed_labels = mask[overshadowed_region].copy()
            
            # Extract bounding box of overshadowed region to rotate it
            y_coords, x_coords = np.where(overshadowed_region)
            if len(y_coords) > 0:
                y_min, y_max = y_coords.min(), y_coords.max() + 1
                x_min, x_max = x_coords.min(), x_coords.max() + 1
                
                # Extract patch and rotate it 180 degrees
                overshadowed_patch = mask[y_min:y_max, x_min:x_max].copy()
                rotated_patch = np.rot90(overshadowed_patch, k=2)  # 180-degree rotation
            
            # Clear the original building position
            noisy_mask[original_region] = 0
            
            # Place shifted building at new position
            noisy_mask[overshadowed_region] = class_id
            
            # Fill original position with rotated overshadowed content
            orig_y_coords, orig_x_coords = np.where(original_region)
            if len(orig_y_coords) > 0 and len(y_coords) > 0:
                orig_y_min, orig_y_max = orig_y_coords.min(), orig_y_coords.max() + 1
                orig_x_min, orig_x_max = orig_x_coords.min(), orig_x_coords.max() + 1
                
                patch_h, patch_w = rotated_patch.shape
                orig_h = orig_y_max - orig_y_min
                orig_w = orig_x_max - orig_x_min
                
                # If sizes match, place rotated patch directly

                if patch_h == orig_h and patch_w == orig_w:
                    noisy_mask[orig_y_min:orig_y_max, orig_x_min:orig_x_max] = rotated_patch
                else:
                    # If sizes don't match, fill with most common overshadowed class
                    if len(overshadowed_labels) > 0:
                        dominant_class = np.bincount(overshadowed_labels).argmax()
                noisy_mask[original_region] = dominant_class
        
        noisy_set[i] = noisy_mask
    return noisy_set
            

In [ ]:
def shift_classes_dominant_fill(y_pred,
                           class_ids,
                           shift_x=5,
                           shift_y=5,
                           shift_prob=1.0,
                           # exclude_classes=None,
                           ):
    """
    Batch-version: shift connected components of classes in `class_ids` across a dataset.
    - y_pred: array-like (N, H, W) of integer label masks
    - class_ids: int or iterable of ints to shift
    - shift_x, shift_y: translation amounts (pixels)
    - shift_prob: per-object probability to apply the shift
    - exclude_classes: iterable of class ids to exclude when choosing fill (optional)
    Returns: noisy_set (same shape as y_pred) with shifts applied.
    """

    y_pred = np.asarray(y_pred)
    if isinstance(class_ids, (int, np.integer)):
        class_ids = [int(class_ids)]
    class_ids = [int(c) for c in class_ids]

    noisy_set = np.copy(y_pred)
    structure = np.ones((3, 3), dtype=np.int32)

    for i, mask in enumerate(y_pred):
        noisy = noisy_set[i].copy()
        orig_mask = mask.copy() # use original for component detection
        occupied = np.zeros_like(mask, dtype=bool) 

        for class_id in class_ids:
            exclude_set = set([class_id])

            class_mask = (orig_mask == class_id)
            if not class_mask.any():
                continue

            labeled, n_objects = ndimage.label(class_mask, structure=structure)

            for obj_id in range(1, n_objects + 1):
                if np.random.rand() > float(shift_prob):
                    continue

                object_mask = (labeled == obj_id).astype(np.float32)

                # Shift the object
                translation_matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
                shifted_object_mask = cv2.warpAffine(object_mask,
                                                     translation_matrix,
                                                     (mask.shape[1], mask.shape[0]),
                                                     flags=cv2.INTER_NEAREST)

                original_region = object_mask == 1
                new_region = shifted_object_mask == 1

                # Clear original region (vacated area)
                noisy[original_region] = 0

                # Place shifted class only where no shifted object exists yet
                noisy[new_region] = class_id

                # Compute neighbors of the vacated area
                neigh_struct = np.ones((3, 3), dtype=bool)
                dilated = ndimage.binary_dilation(original_region, structure=neigh_struct)
                neighbor_mask = dilated & (~original_region)

                # Collect neighbor labels and exclude unwanted classes
                neighbor_labels = noisy[neighbor_mask]
                if neighbor_labels.size == 0:
                    continue

                valid_labels = neighbor_labels[~np.isin(neighbor_labels, list(exclude_set))]
                if valid_labels.size == 0:
                    continue

                counts = np.bincount(valid_labels.astype(np.int64))
                dominant = int(np.argmax(counts))

                # Fill vacated region with dominant adjacent class
                noisy[(original_region == 1) & (new_region == 0)] = dominant

        noisy_set[i] = noisy

    return noisy_set

In [ ]:
# TODO: add random shift to center in noisy zoom, so we have variance in alignement

def noisy_zoom(y_pred, scale=1.05, fill_class=0):
    """
    Zoom label masks in or out.
    - scale > 1.0: zoom in, new border pixels set to fill_class
    - scale < 1.0: zoom out, new border pixels mirrored from the original mask
    """
    y_pred = np.asarray(y_pred)
    single = y_pred.ndim == 2
    if single:
        y_pred = y_pred[None]

    output = np.empty_like(y_pred)
    h, w = y_pred.shape[1], y_pred.shape[2]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, 0.0, scale)

    if scale >= 1.0:
        border_mode = cv2.BORDER_CONSTANT
        border_value = fill_class
    else:
        border_mode = cv2.BORDER_REFLECT_101
        border_value = 0

    for i, mask in enumerate(y_pred):
        output[i] = cv2.warpAffine(
            mask,
            M,
            (w, h),
            flags=cv2.INTER_NEAREST,
            borderMode=border_mode,
            borderValue=border_value,
        )

    return output[0] if single else output

In [6]:
def shift_scene(y_pred, shift_x=5, shift_y=5):
    """
    Shift the full label mask and mirror-fill the newly exposed border pixels.
    y_pred: array-like of shape (N, H, W) or single mask (H, W)
    shift_x, shift_y: pixel translation; can be positive or negative
    """
    y_pred = np.asarray(y_pred)
    single = y_pred.ndim == 2
    if single:
        y_pred = y_pred[None]

    output = np.empty_like(y_pred)
    h, w = y_pred.shape[1], y_pred.shape[2]
    M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])

    for i, mask in enumerate(y_pred):
        output[i] = cv2.warpAffine(
            mask,
            M,
            (w, h),
            flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REFLECT_101,
        )

    return output[0] if single else output

In [ ]:
# Add hierarchy of neighbored fill classes foreach class?

def omission_noise(y_true,
                   class_ids=np.arange(9)[1:], # (4, 5, 8),
                   base_prob=0.5,
                   size_decay=200.0,
                   min_prob=0.01,
                   fallback_class=0):
    """
    Simulate human annotation omission errors for selected object classes.
    Deleted object pixels are filled from the dominant neighboring class.
    """
    y_true = np.asarray(y_true)
    single = y_true.ndim == 2
    if single:
        y_true = y_true[None]

    noisy = y_true.copy()
    structure = np.ones((3, 3), dtype=np.int32)

    for i, mask in enumerate(y_true):
        noisy_mask = noisy[i]
        for class_id in class_ids:
            class_mask = (mask == class_id)
            if not class_mask.any():
                continue

            labeled, n_objects = ndimage.label(class_mask, structure=structure)
            for obj_id in range(1, n_objects + 1):
                obj_mask = (labeled == obj_id)
                obj_size = obj_mask.sum()

                prob = base_prob * np.exp(-obj_size / float(size_decay))
                prob = max(prob, min_prob)

                if np.random.rand() < prob:
                    neighbor_region = ndimage.binary_dilation(obj_mask, structure=structure) & ~obj_mask
                    neighbor_labels = noisy_mask[neighbor_region]

                    if neighbor_labels.size > 0:
                        valid_labels = neighbor_labels[neighbor_labels != class_id]
                        if valid_labels.size == 0:
                            valid_labels = neighbor_labels
                        fill_label = int(np.bincount(valid_labels.astype(np.int64)).argmax())
                    else:
                        fill_label = fallback_class

                    noisy_mask[obj_mask] = fill_label

    return noisy[0] if single else noisy

In [17]:
def add_noise(pr_path, n_classes, noise_method):

    gt_files = sorted(list(GT_PATH.glob("*.tif")))
    y_true = np.array([load_labels(path) for path in gt_files])

    pr_files = sorted(list(pr_path.glob("*.png")))
    y_pred = np.array([load_labels(path) for path in pr_files])
        
    (DATA_ROOT / "noise").mkdir(parents=True, exist_ok=True)
    (DATA_ROOT / "noise" / noise_method).mkdir(parents=True, exist_ok=True)

    # (DATA_ROOT / "noise" / "noisy_test").mkdir(parents=True, exist_ok=True)

    pred_imgs = np.copy(y_true[:2])
    
    # output = add_semantic_swapping(y_true, y_pred, n_classes)
    # output = shift_specific_class(pred_img, shift_x=20, shift_y=20, class_id=8)
    # output = shift_classes_dominant_fill(pred_imgs, shift_x=20, shift_y=20, class_ids=[8])
    # output = noisy_zoom(pred_imgs, scale=.8, fill_class=0)
    # output = shift_scene(pred_imgs, shift_x=20, shift_y=20)
    output = omission_noise(pred_imgs)

    for i, img in enumerate(output):
        Image.fromarray(img).save(DATA_ROOT / "noise" / noise_method / f"TestArea_{{:03d}}.png".format(i+1))

    color_model_output(result_root= DATA_ROOT / "noise" / noise_method)


GT_PATH = DATA_ROOT / "test/labels"
# add_noise(DATA_ROOT / "results", n_classes=9, noise_method="semantic_class_swapping")
# add_noise(DATA_ROOT / "results", n_classes=9, noise_method="class_shifting")
# add_noise(DATA_ROOT / "results", n_classes=9, noise_method="zoom_in")
# add_noise(DATA_ROOT / "results", n_classes=9, noise_method="shift_scene")
add_noise(DATA_ROOT / "test/labels", n_classes=9, noise_method="omission")

Saved colored output to: colored_TestArea_001.png
Saved overlay output to: overlay_TestArea_001.png
Saved colored output to: colored_TestArea_002.png
Saved overlay output to: overlay_TestArea_002.png
